# 🏆 Reranking RAG
## RAG with LangChain + Google Gemini + ChromaDB + CrossEncoder

---

## 🧠 What is Reranking?

Reranking is a **second-pass filtering step** that runs *after* initial retrieval.

**The Two-Stage Architecture:**

```
Stage 1 — Retrieval (fast, approximate)
  Vector similarity search → retrieves top-5 candidates
  Tool: ChromaDB + HuggingFace Embedding model (Bi-Encoder)
  Speed: milliseconds
  Accuracy: good — but similarity ≠ relevance

Stage 2 — Reranking (slower, precise)
  Score each candidate against the question → keep top-2
  Tool: CrossEncoder model  (cross-encoder/ms-marco-MiniLM-L-6-v2)
  Speed: seconds
  Accuracy: much better — directly models relevance
```

**Why do we need both stages?**  
Running a CrossEncoder against *all* chunks in your database would be extremely slow — it processes each (question, chunk) pair together. So we use the fast vector search to narrow down to 5–20 candidates first, then apply the precise CrossEncoder only on those candidates.

**Analogy:**  
Think of a hiring process.
- **Stage 1 (Retrieval)** = HR scans 500 CVs by keyword matching and shortlists 10 candidates. Fast, but keyword matching isn't perfect.
- **Stage 2 (Reranking)** = The hiring manager reads those 10 CVs carefully and ranks the top 2 for interview. Slower, but far more accurate.

You wouldn't ask the hiring manager to read all 500 CVs — that's why you need both stages.

---

## 🔑 Bi-Encoder vs Cross-Encoder — The Core Difference

This is the most important concept in this notebook.

| | Bi-Encoder (retrieval) | Cross-Encoder (reranking) |
|---|---|---|
| **How it works** | Encodes question and chunk *separately* into vectors, then measures distance | Processes question and chunk *together* as a single input |
| **What it produces** | A vector per text (for storage & search) | A relevance score per pair (not a vector) |
| **Speed** | Very fast — vectors pre-computed at index time | Slower — must process each pair at query time |
| **Accuracy** | Good — misses nuanced relevance | Better — sees full interaction between question and chunk |
| **Example** | `all-MiniLM-L6-v2` (HuggingFace) | `ms-marco-MiniLM-L-6-v2` |

```
Bi-Encoder (separate encoding):
  Question  →  [encoder]  →  vector_q
  Chunk     →  [encoder]  →  vector_c
  Score = cosine_similarity(vector_q, vector_c)

Cross-Encoder (joint encoding):
  [Question + Chunk]  →  [encoder]  →  relevance_score
  (processes both together — sees the relationship directly)
```

---

## 🗺️ Full Pipeline Overview

```
┌─────────────────────────────────────────────────────────────┐
│                   INDEXING PHASE (run once)                 │
│  Raw Text → Document → Chunks → HF Embeddings → ChromaDB   │
└─────────────────────────────────────────────────────────────┘
              ↓  (query time — the Reranking flow)
┌─────────────────────────────────────────────────────────────┐
│                   QUERYING PHASE (per question)             │
│                                                             │
│  User Question                                              │
│         │                                                   │
│  [Step 4a] ChromaDB retrieves top-5 candidates (fast)      │
│         │                                                   │
│  [Step 4b] Build (question, chunk) pairs for reranker      │
│         │                                                   │
│  [Step 4c] CrossEncoder scores all 5 pairs (precise)       │
│         │                                                   │
│  [Step 4d] Sort by score → keep top-2                      │
│         │                                                   │
│  [Step 4e] Build prompt with top-2 context                 │
│         │                                                   │
│  [Step 4f] Gemini generates final grounded answer          │
└─────────────────────────────────────────────────────────────┘
```

> 💡 **Notice k=5 in the retriever, top-2 after reranking.**  
> Reranking needs a candidate pool larger than the final count to be useful.  
> Rule of thumb: retrieve 3–5× more than you want to keep, then rerank down.

## 🔄 Progression: Where Reranking Fits

| Approach | Retrieval Strategy | Final Context Quality |
|---|---|---|
| **Basic RAG** | Top-K by vector similarity | Good |
| **Query Rewriting** | Better query → Top-K by similarity | Better |
| **HyDE** | Hypothetical doc → Top-K by similarity | Better |
| **Multi-Query** | N queries → merge → dedup | Broader coverage |
| **Reranking** ⭐ | Top-K by similarity → CrossEncoder scores → Top-M | Highest precision |

> Reranking is **orthogonal** to the other techniques — it can be stacked on top of any of them.  
> Multi-Query + Reranking is a particularly powerful combination: broad coverage first, then precision filtering.

## 📦 Prerequisites & Installation

This notebook runs on **Google Colab** — no local setup needed.

**Stack used:**
- 🔵 **Google Gemini** (`gemini-2.5-flash`) — LLM for answer generation
- 🟠 **HuggingFace Embeddings** (`all-MiniLM-L6-v2`) — Bi-Encoder for vector search
- 🟢 **ChromaDB** — Local in-memory vector store
- 🔴 **CrossEncoder** (`ms-marco-MiniLM-L-6-v2`) — Reranker for precision filtering

**You will need:**
- A **Google Gemini API key** → https://aistudio.google.com/app/apikey (free tier available)

Run the install cell below before anything else.

In [1]:
# ── Install all required packages ────────────────────────────────────────────
# Run this cell first — takes ~60 seconds on Colab
!pip install -q langchain langchain-core langchain-text-splitters langchain-google-genai langchain-chroma langchain-huggingface sentence-transformers chromadb

Could not import runpy._run_module_as_main
Traceback (most recent call last):
  File "<frozen importlib._bootstrap>", line 1371, in _find_and_load
  File "<frozen importlib._bootstrap>", line 1342, in _find_and_load_unlocked
  File "<frozen importlib._bootstrap>", line 938, in _load_unlocked
  File "<frozen importlib._bootstrap>", line 1179, in exec_module
  File "<frozen runpy>", line 14, in <module>
  File "C:\Users\sr43993\AppData\Roaming\uv\python\cpython-3.12-windows-x86_64-none\Lib\importlib\__init__.py", line 57, in <module>
    import warnings
  File "C:\Users\sr43993\AppData\Roaming\uv\python\cpython-3.12-windows-x86_64-none\Lib\warnings.py", line 591, in <module>
    filterwarnings("default", category=DeprecationWarning,
  File "C:\Users\sr43993\AppData\Roaming\uv\python\cpython-3.12-windows-x86_64-none\Lib\warnings.py", line 155, in filterwarnings
    import re
  File "C:\Users\sr43993\AppData\Roaming\uv\python\cpython-3.12-windows-x86_64-none\Lib\re\__init__.py", line 125, 

---
## 🔑 Step 0a — Set Your Gemini API Key

Replace the empty string below with your key from https://aistudio.google.com/app/apikey

> ⚠️ **Never commit your API key to version control.** In production, use environment variables or secret managers.

In [4]:
! pip install langchain_huggingface

Could not import runpy._run_module_as_main
Traceback (most recent call last):
  File "<frozen importlib._bootstrap>", line 1371, in _find_and_load
  File "<frozen importlib._bootstrap>", line 1342, in _find_and_load_unlocked
  File "<frozen importlib._bootstrap>", line 938, in _load_unlocked
  File "<frozen importlib._bootstrap>", line 1179, in exec_module
  File "<frozen runpy>", line 14, in <module>
  File "C:\Users\sr43993\AppData\Roaming\uv\python\cpython-3.12-windows-x86_64-none\Lib\importlib\__init__.py", line 57, in <module>
    import warnings
  File "C:\Users\sr43993\AppData\Roaming\uv\python\cpython-3.12-windows-x86_64-none\Lib\warnings.py", line 591, in <module>
    filterwarnings("default", category=DeprecationWarning,
  File "C:\Users\sr43993\AppData\Roaming\uv\python\cpython-3.12-windows-x86_64-none\Lib\warnings.py", line 155, in filterwarnings
    import re
  File "C:\Users\sr43993\AppData\Roaming\uv\python\cpython-3.12-windows-x86_64-none\Lib\re\__init__.py", line 125, 

In [2]:
import os

# ── Paste your Gemini API key here ───────────────────────────────────────────
os.environ["GOOGLE_API_KEY"] = "AQ.Ab8RN6JII5fO_RhQyE4eASynG6z7-G8taX5DStta8GtteynOFg"   # ← replace with your key

if not os.environ["GOOGLE_API_KEY"]:
    raise ValueError("❌ GOOGLE_API_KEY is not set. Paste your key above before running.")

print("✅ API key set")

✅ API key set


---
## 📚 Step 0b — Imports

| Import | Role |
|---|---|
| `TextLoader` | Reads your `.txt` file from disk after upload |
| `RecursiveCharacterTextSplitter` | Splits text into overlapping chunks |
| `HuggingFaceEmbeddings` | Bi-Encoder — converts text → vectors for storage & search |
| `Chroma` | Local in-memory vector database |
| `ChatGoogleGenerativeAI` | Gemini LLM for answer generation |
| `CrossEncoder` ⭐ | Reranker — scores (question, chunk) pairs for precise relevance |

In [5]:
from langchain_community.document_loaders.text import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_google_genai import ChatGoogleGenerativeAI

# ── ⭐ CrossEncoder from sentence-transformers ────────────────────────────────
# Unlike HuggingFaceEmbeddings (Bi-Encoder), CrossEncoder processes question +
# chunk TOGETHER and outputs a single relevance score — not a vector
from sentence_transformers import CrossEncoder

print("✅ All imports successful")

ModuleNotFoundError: No module named 'langchain_huggingface'

---
## 📄 Step 1 — Upload & Load Your Document

This notebook uses **your own `company_policy.txt`** file via Colab's file upload.

### How to upload:
1. Run the cell below — it opens a file picker
2. Select your `company_policy.txt` file
3. The file is saved to Colab's working directory and loaded automatically

**Why chunk?**  
LLMs have a context window limit. More importantly, smaller chunks = more precise retrieval.  
The retriever finds the *exact paragraph* that answers the question — not the whole document.

> 💡 **Any plain `.txt` file works.** Rename your document to `company_policy.txt`
> before uploading, or change the filename in the code cell below to match yours.

In [ ]:
# ── Step 1a: Upload your file ────────────────────────────────────────────────
# Run this cell → a file picker will appear → select your company_policy.txt
from google.colab import files

print("📂 File picker opening — select your company_policy.txt")
uploaded = files.upload()

# Verify upload
uploaded_filename = list(uploaded.keys())[0]
print(f"\n✅ Uploaded: {uploaded_filename} ({len(uploaded[uploaded_filename])} bytes)")

# ── Step 1b: Load with TextLoader ────────────────────────────────────────────
# TextLoader reads the file and wraps it in a LangChain Document object
# which carries both page_content (the text) and metadata (source path)
print("\nLoading document...\n")

loader = TextLoader(
    uploaded_filename,   # uses the filename exactly as uploaded
    encoding="utf-8"     # always explicit — avoids silent data corruption on special chars
)

documents = loader.load()

print(f"✅ Loaded {len(documents)} document(s)")
print(f"\n📄 Preview (first 300 chars):")
print(documents[0].page_content[:300])
print(f"\n🏷️  Metadata: {documents[0].metadata}")

# ── Step 1c: Split into Chunks ────────────────────────────────────────────────
# chunk_size=200:  max characters per chunk
# chunk_overlap=50: shared characters between adjacent chunks — prevents losing
#                   context when a sentence spans a chunk boundary
print("\nSplitting document into chunks...\n")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print(f"✅ Total Chunks Created: {len(chunks)}")
print(f"\n--- Previewing first 3 chunks ---")
for i, chunk in enumerate(chunks[:3]):
    print(f"\n🔹 Chunk {i+1} ({len(chunk.page_content)} chars):")
    print(chunk.page_content)
    print("-" * 50)

---
## 🔢 Step 2 — Create Embeddings & Vector Store

**HuggingFaceEmbeddings** (`all-MiniLM-L6-v2`) is the Bi-Encoder here:
- Downloads the model locally on first run (~90MB, cached after that)
- Converts each chunk into a 384-dimensional vector
- Runs entirely on CPU — no API key needed

**ChromaDB** stores those vectors in memory and handles similarity search.

> ⚠️ **Notice: `k=5` in the retriever** — not k=2.  
> We deliberately over-retrieve because the reranker needs a candidate pool.  
> Rule of thumb: retrieve 3×–5× more than you want to keep, then rerank down.

In [ ]:
# ── Step 2a: Embedding Model (Bi-Encoder) ────────────────────────────────────
# This is the FAST encoder — each text gets its own vector independently
# Used for initial retrieval only — NOT for the final relevance scoring
print("Loading HuggingFace embedding model...\n")
print("(First run downloads ~90MB — cached for future runs)\n")

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
    # Runs locally — no API key, no data leaving your machine
)

# Sanity check
test_vec = embeddings.embed_query("test")
print(f"✅ Bi-Encoder Ready — vector dimension: {len(test_vec)}")

# ── Step 2b: Build ChromaDB Vector Store ─────────────────────────────────────
print("\nCreating Chroma Vector Store...\n")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings
    # No persist_directory on Colab — in-memory is fine for demos
)

print(f"✅ Vector Store Ready — {vectorstore._collection.count()} vectors stored")

# ── Step 2c: Create Retriever — k=5 (pool for reranking) ─────────────────────
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 5}   # Over-retrieve: more candidates = better reranking
)

print("\n✅ Retriever Ready (k=5 candidates → will be reranked to top-2)")

---
## 🤖 Step 3a — Load the LLM (Gemini)

`gemini-2.5-flash` is fast, capable, and has a generous free tier.

**Key parameter: `temperature=0`**  
Always use 0 for RAG — deterministic output means the answer is grounded in the retrieved context, not creative variations.

In [ ]:
# ── LLM: Gemini 2.5 Flash ────────────────────────────────────────────────────
print("Loading LLM...\n")

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0   # Must be 0 for RAG — deterministic, grounded answers only
)

print("✅ Gemini LLM Ready")

---
## 🏆 Step 3b — Load the CrossEncoder Reranker

### What is `cross-encoder/ms-marco-MiniLM-L-6-v2`?

A fine-tuned transformer model trained on the MS MARCO passage ranking dataset.
It takes a `[question, passage]` pair as input and outputs a **single float score** — higher = more relevant.

```
Input:   ["What is the maternity leave policy?",
          "Employees are eligible for 6 months maternity leave."]

Output:  +8.42   ← high relevance score

Input:   ["What is the maternity leave policy?",
          "Overtime is compensated at 1.5x the hourly rate."]

Output:  -5.17  ← low relevance score (not relevant at all)
```

### Why not just use the embedding similarity scores?

```
Scenario: "What is the notice period?"

Bi-Encoder might rank this highly:
  "Notice boards are located on each floor..."   (contains "notice" — topically adjacent)

CrossEncoder correctly demotes it:
  Low score — it reads both and understands "notice period" ≠ "notice boards"
```

> 💡 First run downloads ~68MB from HuggingFace — subsequent runs use the Colab cache.

In [ ]:
# ── CrossEncoder: processes (question, chunk) pairs jointly ──────────────────
# Unlike HuggingFaceEmbeddings (Bi-Encoder), CrossEncoder produces a relevance
# score per pair — NOT a vector. It sees both texts together.
print("Loading CrossEncoder Reranker...\n")
print("(First run downloads ~68MB from HuggingFace — cached for future runs)\n")

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

print("✅ CrossEncoder Reranker Ready")

# ── Sanity Check: See raw CrossEncoder scores on sample pairs ─────────────────
# This lets you SEE what the scores look like before querying your document
sample_question = "What is the maternity leave policy?"
sample_pairs = [
    [sample_question, "Employees are eligible for 6 months maternity leave."],
    [sample_question, "The office is located on the 3rd floor of Building B."],
    [sample_question, "Annual leave must be applied for two weeks in advance."],
]

sample_scores = reranker.predict(sample_pairs)

print(f"\n🧪 Sample CrossEncoder Scores (question: '{sample_question}'):")
for pair, score in zip(sample_pairs, sample_scores):
    relevance = "✅ Relevant" if score > 0 else "❌ Not relevant"
    print(f"\n  Score: {score:+.4f}  {relevance}")
    print(f"  Passage: {pair[1][:80]}")

---
## 💬 Step 4 — Interactive Q&A Loop with Reranking

### The Full Flow for Every Question:

```
User question
      │
  ChromaDB retrieves top-5 candidates (Bi-Encoder similarity)
      │
  Build pairs: [(question, chunk1), (question, chunk2), ..., (question, chunk5)]
      │
  CrossEncoder scores all 5 pairs → [+8.4, -1.2, +6.7, +2.1, -3.5]
      │
  Sort by score descending → [(chunk1, +8.4), (chunk3, +6.7), ...]
      │
  Keep top-2 → [(chunk1, +8.4), (chunk3, +6.7)]
      │
  Build context from top-2 real chunks → Gemini → answer
```

### What to observe while running:
- The **initial retrieval order** (by vector similarity) vs the **reranked order** (by CrossEncoder score)
- Chunks that scored low — notice they're often *topically related* but not *directly answering* the question
- Negative scores mean the CrossEncoder judged the chunk as *not relevant at all*
- How the top-2 after reranking compares to the top-2 from initial retrieval — they often differ

**Try these questions:**
- `What is the maternity leave policy?`
- `How many days notice do I need to give?`
- `Can I work from home?`
- `What is the overtime rate?`

In [ ]:
print("Reranking RAG System Ready. Type 'exit' to stop.\n")
print("=" * 60)

while True:

    question = input("\n❓ Ask a question (or type 'exit'): ")

    if question.lower() == "exit":
        print("\n👋 Goodbye!")
        break

    print("\n===== ❓ USER QUESTION =====")
    print(question)

    # ── STEP 4a: RETRIEVE TOP-5 CANDIDATES (Bi-Encoder) ──────────────────────
    # Fast vector similarity search — ordered by cosine distance, not true relevance
    retrieved_docs = retriever.invoke(question)

    print("\n===== 📄 RETRIEVED CHUNKS (before reranking — vector similarity order) =====")
    for index, doc in enumerate(retrieved_docs):
        print(f"\n  Candidate {index + 1}: {doc.page_content}")

    # ── STEP 4b: BUILD (QUESTION, CHUNK) PAIRS ───────────────────────────────
    # CrossEncoder needs both texts together — it cannot process them separately
    # This is the fundamental difference from Bi-Encoder which encodes separately
    pairs = [
        [question, doc.page_content]
        for doc in retrieved_docs
    ]

    # ── STEP 4c: SCORE ALL PAIRS WITH CROSSENCODER ───────────────────────────
    # reranker.predict() processes all pairs in one batch call
    # Returns a numpy array of floats — one score per pair
    # Higher score = more relevant to the question
    scores = reranker.predict(pairs)

    print("\n===== 📊 CROSSENCODER SCORES (raw relevance scores per chunk) =====")
    for index in range(len(retrieved_docs)):
        relevance_label = "✅" if scores[index] > 0 else "❌"
        print(f"\n  {relevance_label} Candidate {index + 1}  |  Score: {scores[index]:+.4f}")
        print(f"  {retrieved_docs[index].page_content[:120]}")

    # ── STEP 4d: SORT BY SCORE AND KEEP TOP-2 ────────────────────────────────
    # zip() pairs each doc with its score → sort by score (desc) → slice top-2
    scored_docs = list(zip(retrieved_docs, scores))

    sorted_docs = sorted(
        scored_docs,
        key=lambda item: item[1],   # sort by score (index 1 of each tuple)
        reverse=True                # highest score first
    )

    top_docs = sorted_docs[:2]   # keep only the 2 most relevant

    print("\n===== 🏆 TOP-2 AFTER RERANKING =====")
    for index, (doc, score) in enumerate(top_docs):
        print(f"\n  Rank {index + 1}  |  Score: {score:+.4f}")
        print(f"  {doc.page_content}")

    # ── STEP 4e: BUILD CONTEXT FROM TOP-2 RERANKED CHUNKS ────────────────────
    # Only the most relevant chunks go into the prompt — no noise from lower-ranked ones
    context = "\n\n".join(
        doc.page_content for doc, score in top_docs
    )

    # ── STEP 4f: CRAFT FINAL ANSWER PROMPT ───────────────────────────────────
    # Original question + real context = grounded, trustworthy answer
    prompt = f"""You are a helpful assistant.

Answer ONLY using the provided context.
If the answer is not present in the context, reply exactly with:
"I could not find that information in the document."

Context:
{context}

Question:
{question}
"""

    # ── STEP 4g: GENERATE ANSWER ──────────────────────────────────────────────
    response = llm.invoke(prompt)

    print("\n===== 🤖 FINAL ANSWER =====")
    print(response.content)

---
## 🧪 Bonus — Single-Shot Reranking Query

Test any question without entering the interactive loop.  
Change the `question` variable and re-run this cell.

In [ ]:
# ── Change this question and re-run ──────────────────────────────────────────
question = "What is the notice period for resignation?"

# Retrieve top-5 candidates
retrieved_docs = retriever.invoke(question)

# Score with CrossEncoder
pairs = [[question, doc.page_content] for doc in retrieved_docs]
scores = reranker.predict(pairs)

# Sort and keep top-2
top_docs = sorted(zip(retrieved_docs, scores), key=lambda x: x[1], reverse=True)[:2]

# Build context and generate answer
context = "\n\n".join(doc.page_content for doc, _ in top_docs)
prompt = f"""Answer ONLY using the context below.
If not found, say: "I could not find that information in the document."

Context:
{context}

Question: {question}
"""
answer = llm.invoke(prompt).content

print(f"❓ Question: {question}")
print(f"\n📊 Top-2 after reranking:")
for i, (doc, score) in enumerate(top_docs, 1):
    print(f"  Rank {i} (score {score:+.4f}): {doc.page_content[:100]}")
print(f"\n🤖 Answer: {answer}")

---
## 🔬 Bonus — Reranking Impact: Before vs After

This cell runs multiple questions and shows how reranking changes the *order* of chunks.  
The best demo for training audiences — makes the ranking shift visible without any manual testing.

In [ ]:
test_questions = [
    "What is the maternity leave policy?",
    "What is the notice period?",
    "Can I work from home?",
    "What is the performance bonus?",
]

for q in test_questions:
    print(f"\n{'='*65}")
    print(f"❓ {q}")

    # Retrieve and score
    docs = retriever.invoke(q)
    pairs = [[q, doc.page_content] for doc in docs]
    scores = reranker.predict(pairs)

    # Before reranking (vector similarity order)
    print(f"\n  📄 Before reranking (vector similarity order):")
    for i, (doc, score) in enumerate(zip(docs, scores)):
        print(f"    {i+1}. [score {score:+.2f}]  {doc.page_content[:70]}...")

    # After reranking (CrossEncoder order)
    reranked = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)
    print(f"\n  🏆 After reranking (CrossEncoder relevance order):")
    for i, (doc, score) in enumerate(reranked[:2]):
        print(f"    {i+1}. [score {score:+.2f}]  {doc.page_content[:70]}...")

    # Did the top-1 change?
    original_top = docs[0].page_content
    reranked_top = reranked[0][0].page_content
    changed = original_top != reranked_top
    print(f"\n  {'⭐ Top chunk CHANGED after reranking' if changed else '= Top chunk unchanged'}")

---
## 🎯 Key Takeaways

| Concept | What to Remember |
|---|---|
| **Two-stage retrieval** | Fast Bi-Encoder first (candidates), precise CrossEncoder second (ranking) |
| **Bi-Encoder** | Encodes texts *separately* → vectors → cosine similarity |
| **CrossEncoder** | Encodes question + chunk *together* → direct relevance score |
| **k=5 retrieve, top-2 keep** | Always over-retrieve — reranking needs a pool to reorder |
| **Scores can be negative** | CrossEncoder output is unbounded — negative = not relevant |
| **Reranking is orthogonal** | Stacks on top of any previous technique (Multi-Query + Reranking is powerful) |

---

## ⚖️ Reranking Trade-offs

| ✅ Strengths | ⚠️ Weaknesses |
|---|---|
| Highest precision — finds truly relevant chunks | CrossEncoder adds latency (runs on CPU by default) |
| Catches cases where vector similarity misleads | Needs a candidate pool — useless with k=1 retrieval |
| Scores are interpretable (higher = more relevant) | Larger models needed for multilingual reranking |
| No LLM call needed — CrossEncoder is lightweight | First-run model download (~68MB) |

---

## 🔄 Complete RAG Techniques — Final Summary

```
Basic RAG:
  question ──────────────────────────────────────→ top-K → answer

Query Rewriting:
  question → rewrite ────────────────────────────→ top-K → answer

HyDE:
  question → hypothetical doc ───────────────────→ top-K → answer

Multi-Query:
  question → 4 variants → 4x retrieve → dedup ──→ top-K → answer

Reranking:
  question ─────────────────→ top-5 → CrossEncoder → top-2 → answer
                                       (reorder by true relevance)

Power Combo (Multi-Query + Reranking):
  question → 4 variants → 4x retrieve → dedup → CrossEncoder → top-2 → answer
  (broadest coverage + highest precision)
```

---

## 🚀 Next Steps to Explore

1. **Multi-Query + Reranking** — Combine both notebooks: generate 4 queries, merge chunks, rerank the full pool
2. **`bge-reranker-large`** — Higher accuracy at the cost of speed, worth it for production
3. **GPU acceleration** — Pass `device='cuda'` to `CrossEncoder()` for 10–20× faster scoring
4. **Cohere Rerank API** — Managed reranking service, excellent quality, costs per call
5. **RAGAS evaluation** — Measure whether reranking actually improves faithfulness and precision vs vanilla retrieval